# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided as a Croissant schema by URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`. This allows us to inspect the entire Croissant schema, including Record Sets, Fields, and other attributes.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Published:', getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview

List and review available **Record Sets** and their Fields. All entities are referenced by their `@id` fields for clarity and reproducibility.


In [ ]:
# List record sets and their IDs
print('Record Sets:')
record_sets = []
for rs in metadata.record_sets:
    recset_id = rs.id
    recset_name = getattr(rs, 'name', '')
    print(f"  - Name: {recset_name}, @id: {recset_id}")
    record_sets.append(recset_id)

# Show fields for each record set
for rs in metadata.record_sets:
    recset_id = rs.id
    recset_name = getattr(rs, 'name', '')
    print(f"\nFields for RecordSet '{recset_name}' (@id: {recset_id}):")
    for field in rs.fields:
        field_id = field.id
        field_name = getattr(field, 'name', '')
        data_type = getattr(field, 'data_type', '')
        print(f"  - Field Name: {field_name}, @id: {field_id}, Type: {data_type}")

## 3. Data Extraction

Let's extract the data from the main Record Set containing protein data, using its `@id`. All `@id` values are discovered from the overview above. We demonstrate loading *all* available record sets into DataFrames, then preview the main table.


In [ ]:
# For demonstration we will list all 'record_sets' by their @id
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for RecordSet @id: {record_set_id}")

# Pick the main record set (typically with most fields/rows)
if record_sets:
    main_rs_id = record_sets[0]
    main_df = dataframes[main_rs_id]
    print(f"Columns in main RecordSet (@id={main_rs_id}):\n", main_df.columns.tolist())
    display(main_df.head())
else:
    print('No record sets found.')

## 4. Exploratory Data Analysis (EDA)

Basic data cleaning and EDA: We'll filter, normalize, and group using some representative numeric and categorical fields. 

Set the field `@id`s (as variables) you want to analyze, based on the output of code above. Adjust these if needed according to your schema's main `@id`s.


In [ ]:
# Update these variables based on your schema (see previous cell's output)
record_set_id = main_rs_id
df = dataframes[record_set_id]

# Choose a numeric field (by @id) for filtering and normalization
# Example: for a proteomics table, 'coverage_percent' or 'peptide_count' might be a numeric field @id
numeric_field = None
for col in df.columns:
    if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # Try to use a common field name for demonstration
    numeric_field = df.columns[0]

threshold = 10  # Example threshold
if numeric_field in df.columns:
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == object:
            group_field = col
            break
    if group_field:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        )
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print('No numeric field available for EDA.')

## 5. Visualization

Let's plot the distribution of the selected numeric field, and visualize the group means if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Plot histogram of numeric_field
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='muted')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we:

- Loaded and described a FAIR Croissant dataset using `mlcroissant`.
- Discovered all record sets, fields, and their proper `@id` identifiers.
- Extracted the main table, explored numeric/categorical fields, and applied simple data processing (filtering, normalization, grouping).
- Visualized distributions and group means.

For deeper analysis, consult the dataset schema for more field details and extend the code to explore additional relationships or advanced analytics.
